# Elite MLX Training Pipeline (Apple Silicon)
Includes batching, augmentation, metrics, checkpointing, and training curves.

In [ ]:

# !pip install mlx pillow scikit-learn matplotlib

import mlx.core as mx
import mlx.nn as nn
import mlx.optimizers as optim

import os, random
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt


In [ ]:

# CONFIG
IMG_SIZE = 64
BATCH_SIZE = 32
EPOCHS = 15
DATASET_PATH = "data/fruits"


In [ ]:

# AUGMENTATION
def augment(img):
    if random.random() > 0.5:
        img = np.fliplr(img)
    if random.random() > 0.5:
        img = np.flipud(img)
    return img


In [ ]:

# LOAD DATA
def load_images(path):
    X, y = [], []
    classes = sorted(os.listdir(path))

    for i, c in enumerate(classes):
        p = os.path.join(path, c)
        if not os.path.isdir(p): continue
        for f in os.listdir(p):
            try:
                img = Image.open(os.path.join(p, f)).convert("RGB")
                img = img.resize((IMG_SIZE, IMG_SIZE))
                img = np.array(img) / 255.0
                X.append(img)
                y.append(i)
            except:
                pass
    return np.array(X), np.array(y), classes

X, y, classes = load_images(DATASET_PATH)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2)

X_train, X_val = mx.array(X_train), mx.array(X_val)
y_train, y_val = mx.array(y_train), mx.array(y_val)


In [ ]:

# BATCH GENERATOR
def get_batches(X, y, batch_size):
    idx = np.random.permutation(len(X))
    for i in range(0, len(X), batch_size):
        batch_idx = idx[i:i+batch_size]
        yield X[batch_idx], y[batch_idx]


In [ ]:

# MODEL
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(3,32,3), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32,64,3), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64,128,3), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(128,256,3), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.flatten = nn.Flatten()
        self.fc = nn.Sequential(
            nn.Linear(256*2*2,512), nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(512,len(classes))
        )
    def __call__(self,x):
        return self.fc(self.flatten(self.conv(x)))

model = CNN()


In [ ]:

# LOSS + METRICS
def loss_fn(model, X, y):
    logits = model(X)
    return mx.mean(nn.losses.cross_entropy(logits, y.astype(mx.int32)))

def accuracy(model, X, y):
    preds = mx.argmax(model(X), axis=1)
    return mx.mean((preds == y).astype(mx.float32))


In [ ]:

# TRAIN LOOP + CHECKPOINT
optimizer = optim.Adam(1e-3)

train_losses, val_accuracies = [], []

for epoch in range(EPOCHS):
    epoch_loss = 0

    for xb, yb in get_batches(X_train, y_train, BATCH_SIZE):
        loss, grads = mx.value_and_grad(model, loss_fn)(model, xb, yb)
        optimizer.update(model, grads)
        epoch_loss += loss.item()

    val_acc = accuracy(model, X_val, y_val).item()

    train_losses.append(epoch_loss)
    val_accuracies.append(val_acc)

    # Save checkpoint
    mx.savez(f"checkpoint_epoch_{epoch}.npz", *model.parameters())

    print(f"Epoch {epoch+1}: Loss={epoch_loss:.3f}, Val Acc={val_acc:.3f}")


In [ ]:

# PLOTS
plt.figure()
plt.plot(train_losses)
plt.title("Training Loss")

plt.figure()
plt.plot(val_accuracies)
plt.title("Validation Accuracy")
plt.show()
